# Persona opinion distributions from open-weight instruct models (CIS 3194)

For every prompt (18 personas × 3 items × {es, en} × {original, reversed} option order), this notebook reads the model's probability of each answer label in **one forward pass**, with no sampling.

`VARIANT` picks the label style: `"numbers"` (`prompts.jsonl`, options 1, 2, 3…) or `"letters"` (`prompts_letters.jsonl`, options A, B, C…). The letters variant is identical apart from the labels and the word "número"/"letra" in the instruction; it tests whether order effects come from the answer token rather than the option content. Both variants write CSVs with the same columns.

1. The prompt goes in as a **single user turn** through the model's chat template.
2. **P(label L)** = probability that the reply *starts with* L. The candidate tokens for L are every token that decodes to exactly L apart from whitespace. In Mistral that means `1`/`<0x31>` for digits (there is no `▁1`, so digit replies begin with a separate space token `▁`), and `A`/`▁A`/`<0x41>` for letters. Letters are **uppercase only**: "a", "e" and "o" are Spanish words and "a" is the English article. So P(L) = P(L first) + P(`▁` first) · P(L | `▁`), with both terms from the same forward pass over *prompt + `▁`*.
3. The probabilities are **renormalized** over the valid option labels and **mapped back to CIS codes** via `number_to_code` / `label_to_code`.
4. The two option orders are **averaged**, giving one row per persona × item × language.

`valid_mass` records how much probability landed on valid option labels before renormalizing. Values near 1 mean the model followed the format; low values mean the distribution rests on little evidence.

**Setup:** Runtime → Change runtime type → **T4 GPU**. Set `VARIANT`, then run all cells; you'll be asked to upload `data/processed/prompts.jsonl` or `prompts_letters.jsonl`.

**Models:** set `MODEL_KEY` in the config cell ("mistral" = Mistral-7B-Instruct-v0.3, France; "qwen" = Qwen2.5-7B-Instruct, China). The token details below were written for Mistral; the sanity-check cell verifies that the other tokenizers map option labels correctly before the full run. Each chat template may add its own default system prompt (e.g. Qwen's), which is part of the model as deployed.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
# Pick one model per run; outputs are named after the model, so runs don't overwrite each other.
MODELS = {
    "mistral": "mistralai/Mistral-7B-Instruct-v0.3",   # France
    "qwen":    "Qwen/Qwen2.5-7B-Instruct",             # China
}
MODEL_KEY = "mistral"
MODEL_ID = MODELS[MODEL_KEY]
VARIANT = "numbers"   # "numbers" (options 1, 2, 3...) or "letters" (options A, B, C...)
ORDERS = "two"        # "two" (original + reversed) or "cyclic" (all rotations; built with build_prompts.py --orders cyclic)

PROMPTS_PATH = "prompts" + ("_cyclic" if ORDERS == "cyclic" else "") + ("_letters" if VARIANT == "letters" else "") + ".jsonl"
MODEL_TAG = MODEL_ID.split("/")[-1]
# numbers + two orders keeps the original file names
SUFFIX = ("cyclic_" if ORDERS == "cyclic" else "") + ("letters_" if VARIANT == "letters" else "")
OUT_PROMPT_LEVEL = f"model_prompt_level_{SUFFIX}{MODEL_TAG}.csv"
OUT_DISTRIBUTIONS = f"model_distributions_{SUFFIX}{MODEL_TAG}.csv"

In [ ]:
import json, os

if not os.path.exists(PROMPTS_PATH):
    from google.colab import files
    files.upload()   # choose data/processed/<PROMPTS_PATH>

with open(PROMPTS_PATH, encoding="utf-8") as f:
    prompts = [json.loads(line) for line in f]


def label_to_code(record):
    # displayed label ("1"/"A", top to bottom) -> CIS code
    return record.get("label_to_code") or record["number_to_code"]


assert all(("label_to_code" in p) == (VARIANT == "letters") for p in prompts), "VARIANT doesn't match the prompts file"
print(len(prompts), "prompts from", PROMPTS_PATH)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),  # fp16 (~14.5 GB) doesn't fit safely on a T4
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

### Answer tokens
`LABEL_IDS[L]` = every vocabulary token that decodes to exactly the label L (case-sensitive), with or without surrounding whitespace, including byte-fallback tokens. `SPACE_ID` = the bare space token a reply may start with. Check the printout: each label should have at least one token, and for Mistral letters you should see `<0x41>`, `▁A` and `A`.

In [ ]:
LABELS = sorted({label for p in prompts for label in label_to_code(p)})

decoded = tok.batch_decode([[i] for i in range(len(tok))])
LABEL_IDS = {L: [i for i, s in enumerate(decoded) if s.strip() == L] for L in LABELS}
SPACE_ID = next(tok.convert_tokens_to_ids(t) for t in ("▁", "Ġ") if t in tok.get_vocab())

for L, ids in LABEL_IDS.items():
    print(L, ids, tok.convert_ids_to_tokens(ids))
print("space token:", SPACE_ID, repr(tok.convert_ids_to_tokens(SPACE_ID)))
assert all(LABEL_IDS.values()), "some option label has no token"

In [ ]:
def chat_ids(prompt):
    enc = tok.apply_chat_template(
        [{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    return enc["input_ids"]


@torch.no_grad()
def label_probs(prompt, labels):
    # P(reply starts with L) for each option label L, unnormalized
    ids = torch.cat([chat_ids(prompt), torch.tensor([[SPACE_ID]])], dim=1).to(model.device)
    logits = model(ids).logits[0, -2:].float()
    p_first, p_after_space = torch.softmax(logits, dim=-1)
    return {
        L: (p_first[LABEL_IDS[L]].sum() + p_first[SPACE_ID] * p_after_space[LABEL_IDS[L]].sum()).item()
        for L in labels
    }


def score(record):
    # CIS code -> renormalized probability, plus the raw mass on valid options
    mapping = label_to_code(record)
    raw = label_probs(record["prompt"], list(mapping))
    valid_mass = sum(raw.values())
    return {mapping[L]: p / valid_mass for L, p in raw.items()}, valid_mass

### Sanity check before the full run
For a few prompts (both languages, both orders), check that the **token IDs the model actually generates** are among the candidates counted in `LABEL_IDS` (or the space token followed by one). Also look at how the probability of each label splits across its token forms (e.g. `A` vs `▁A` vs `▁`+`A`): this shows which form Mistral really uses at the answer position. If the check fails, fix it before running everything; otherwise the CSV would come out without errors but with probabilities assigned to the wrong labels.

In [ ]:
CANDIDATES = {i for ids in LABEL_IDS.values() for i in ids}


@torch.no_grad()
def token_breakdown(prompt, labels):
    # probability of each label split by token form: first-position tokens, and space-then-label
    ids = torch.cat([chat_ids(prompt), torch.tensor([[SPACE_ID]])], dim=1).to(model.device)
    p_first, p_after_space = torch.softmax(model(ids).logits[0, -2:].float(), dim=-1)
    rows = []
    for L in labels:
        row = {"label": L, "L": 0.0, "▁L": 0.0, "byte": 0.0}
        for i in LABEL_IDS[L]:
            t = tok.convert_ids_to_tokens(i)
            form = "byte" if t.startswith("<0x") else "▁L" if t[0] in "▁Ġ" else "L"
            row[form] += p_first[i].item()
        row["▁ then L"] = (p_first[SPACE_ID] * p_after_space[LABEL_IDS[L]].sum()).item()
        rows.append(row)
    return pd.DataFrame(rows).set_index("label")


import pandas as pd

checks = [p for p in prompts if p["persona_id"] == prompts[0]["persona_id"]]   # one persona: 3 items x 2 langs x 2 orders
all_ok = True
for r in checks:
    ids = chat_ids(r["prompt"]).to(model.device)
    out = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=4, do_sample=False, pad_token_id=tok.eos_token_id)
    gen = out[0, ids.shape[1]:].tolist()
    answer_tok = gen[1] if gen[0] == SPACE_ID and len(gen) > 1 else gen[0]
    ok = answer_tok in CANDIDATES
    all_ok &= ok
    raw = label_probs(r["prompt"], list(label_to_code(r)))
    top = max(raw, key=raw.get)
    print(f"{r['prompt_id']:<24} generated {gen} {tok.convert_ids_to_tokens(gen)} -> {'OK' if ok else 'NOT A CANDIDATE'}"
          f" | top label by probability: {top} ({raw[top] / sum(raw.values()):.2f}), valid mass {sum(raw.values()):.3f}")

print("\nProbability split by token form, first prompt (L = no leading space, ▁L = with space, byte = byte fallback, ▁ then L = separate space token first):")
print(token_breakdown(checks[0]["prompt"], list(label_to_code(checks[0]))).round(4).to_string())
assert all_ok, "the model generated a token that is not counted as an answer label: check LABEL_IDS before the full run"

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

rows = []
for r in tqdm(prompts):
    dist, valid_mass = score(r)
    for code_, p in dist.items():
        rows.append({
            "prompt_id": r["prompt_id"], "persona_id": r["persona_id"], "item": r["item"],
            "lang": r["lang"], "order": r["order"], "code": code_, "proportion": p, "valid_mass": valid_mass,
        })
prompt_level = pd.DataFrame(rows)
prompt_level.insert(0, "model", MODEL_ID)
prompt_level.to_csv(OUT_PROMPT_LEVEL, index=False)
prompt_level.groupby(["item", "lang"])["valid_mass"].describe()[["mean", "min"]]

### Average the option orders
One row per persona × item × language. `p_1`…`p_6` are proportions by **CIS code** (empty for codes the item doesn't have), directly comparable to `P42_1`… in `personas.csv`. `order_tv` measures how much the answer depends on option order (0 = no effect, 1 = completely different); with the two default orders it is the total-variation distance between them, and with cyclic orders it is twice the mean distance of each order's distribution from the average. `n_orders` is the number of orders averaged.

In [ ]:
by_order = prompt_level.pivot_table(
    index=["persona_id", "item", "lang", "code"], columns="order", values="proportion"
)
# one column per option order; each item uses its own set, so other items' orders are NaN and skipped
by_order["proportion"] = by_order.mean(axis=1)

wide = by_order["proportion"].unstack("code")
wide.columns = [f"p_{c}" for c in wide.columns]

# order_tv = 2 x mean over orders of TV(order distribution, average distribution).
# With two orders this equals the TV between them (the original definition).
orders = by_order.drop(columns="proportion")
dev = orders.sub(by_order["proportion"], axis=0).abs()
tv_per_order = dev.groupby(level=["persona_id", "item", "lang"]).sum() / 2
wide["order_tv"] = 2 * tv_per_order.mean(axis=1)

mass = prompt_level.drop_duplicates("prompt_id").pivot_table(
    index=["persona_id", "item", "lang"], columns="order", values="valid_mass"
)
wide["valid_mass_original"] = mass["original"]
wide["valid_mass_reversed"] = mass["reversed"]
wide["valid_mass_min"] = mass.min(axis=1)
wide["n_orders"] = mass.notna().sum(axis=1)

distributions = wide.reset_index()
distributions.insert(0, "model", MODEL_ID)
distributions.to_csv(OUT_DISTRIBUTIONS, index=False)
print(len(distributions), "rows")
distributions.head(6).round(3)

In [ ]:
print("Mean order effect (TV) by item x language:")
print(distributions.groupby(["item", "lang"])["order_tv"].mean().round(3).unstack())

In [ ]:
from google.colab import files
files.download(OUT_DISTRIBUTIONS)
files.download(OUT_PROMPT_LEVEL)